# [문제 정의] 수락률 분석 및 문제점 도출

In [ ]:
df['event_time'] = pd.to_datetime(df['time'], unit='s')

In [ ]:
# ---------------------------------------------------------
# [방법 1] 필터링 & 정렬 (가장 직관적인 형태)
# ---------------------------------------------------------

# 1. 우리가 관심 있는 이벤트 목록 정의
target_events = [
    'editor_run_paraphrasing',
    'run_paraphrasing',
    'editor_selected_paraphrasing',
    'selected_paraphrasing'
]

# 2. 해당 이벤트만 남기기 (Row Filtering)
filtered_df = df[df['event_name'].isin(target_events)].copy()

# 3. 보고 싶은 컬럼만 남기기 (Column Selection)
target_columns = [
    'distinct_id_event',                                # 사용자 ID
    'event_name',                                       # 이벤트명 (Run/Select)
    'llm_name',                                         # 모델명
    'maintenance',                                      # 교정 강도
    'field',                                            # 글 분야
    'total_paraphrasing_sentence_count',                # 동일 문장으로 재실행한 수(3의 배수로 측정: 3-1회 실행, 6-2회 실행)
    'response_time_seconds',                            # 소요 시간
    'event_time'                                        # timestamp
]

final_df = filtered_df[target_columns]

# apply & lambda 활용하여 3으로 나누기
col_name = 'total_paraphrasing_sentence_count'
if col_name in final_df.columns:
    # 숫자인 경우 3으로 나누고, 비어있거나(NaN) 숫자가 아니면 그대로 둠(pass)
    final_df[col_name] = final_df[col_name].apply(
        lambda x: x / 3 if pd.notnull(x) and isinstance(x, (int, float)) else x
    )

# 4. 사용자별(distinct_id_event)로 모아서 보기 좋게 정렬
# 시간 순서(timestamp)가 있다면 같이 정렬해야 흐름이 보입니다.
sort_cols = ['distinct_id_event']
if 'event_time' in df.columns:
    sort_cols.append('event_time')

final_df = final_df.sort_values(by=sort_cols)

In [ ]:
final_df

In [ ]:
final_df.to_csv('./data/merge_acceptance_final.csv', index=False)

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import platform
import seaborn as sns
sns.set_theme(style="darkgrid")
import koreanize_matplotlib

In [ ]:
df = pd.read_csv('./data/merge_acceptance_final.csv')

In [ ]:
df

In [ ]:
df['maintenance'].value_counts()

In [ ]:
df[df['maintenance'] == 'strong']['llm_name'].value_counts()

In [ ]:
df[df['maintenance'] == 'moderate']['llm_name'].value_counts()

In [ ]:
df[df['maintenance'] == 'weak']['llm_name'].value_counts()

In [ ]:
pd.crosstab(df['llm_name'], df['field'])

In [ ]:
pd.crosstab(df['llm_name'], df['maintenance'])

In [ ]:
# ---------------------------------------------------------
# 1. 데이터 로드
# ---------------------------------------------------------
df_field = pd.read_csv('llm_name_field_crosstab.csv')
df_maintenance = pd.read_csv('llm_name_maintenance_crosstab.csv')

# ---------------------------------------------------------
# 2. 데이터 전처리 (Wide -> Long Format 변환)
# ---------------------------------------------------------
# Seaborn에서 그리기 좋게 데이터 형태를 변경합니다.
df_field_long = df_field.melt(id_vars='llm_name', var_name='Field', value_name='Count')
df_maint_long = df_maintenance.melt(id_vars='llm_name', var_name='Maintenance', value_name='Count')

# ---------------------------------------------------------
# 3. 시각화 설정 (한글 폰트 및 스타일)
# ---------------------------------------------------------
plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows 한글 폰트
plt.rcParams['axes.unicode_minus'] = False     # 마이너스 기호 깨짐 방지
sns.set_theme(style="whitegrid", font='Malgun Gothic')

# 그래프 그리기 함수
def plot_distribution(data, x_col, hue_col, title, filename):
    plt.figure(figsize=(12, 6))

    # 막대 그래프 생성
    ax = sns.barplot(
        data=data,
        x=x_col,
        y='Count',
        hue=hue_col,
        palette='viridis',  # 색상 테마
        edgecolor='black'   # 막대 테두리
    )

    # 핵심: 데이터 불균형이 심할 경우 로그 스케일 적용 (필요시 주석 해제)
    # ax.set_yscale('log')

    # 타이틀 및 라벨 설정
    plt.title(title, fontsize=16, pad=20, fontweight='bold')
    plt.xlabel(x_col, fontsize=12)
    plt.ylabel('사용 횟수', fontsize=12)
    plt.legend(title='LLM Model', bbox_to_anchor=(1.05, 1), loc='upper left')

    # 값 표시 (숫자가 너무 겹치면 제거 가능)
    # for p in ax.patches:
    #     if p.get_height() > 0:
    #         ax.annotate(f'{int(p.get_height())}',
    #                     (p.get_x() + p.get_width() / 2., p.get_height()),
    #                     ha='center', va='bottom', fontsize=9, color='black', xytext=(0, 5), textcoords='offset points')

    plt.tight_layout()
    plt.show()
    # plt.savefig(filename, dpi=300) # 저장 필요시 사용

# ---------------------------------------------------------
# 4. 그래프 출력
# ---------------------------------------------------------
# (1) Field별 LLM 분포
plot_distribution(
    df_field_long,
    x_col='Field',
    hue_col='llm_name',
    title='분야(Field)별 LLM 모델 배정 현황',
    filename='field_distribution.png'
)

# (2) 교정강도(Maintenance)별 LLM 분포
plot_distribution(
    df_maint_long,
    x_col='Maintenance',
    hue_col='llm_name',
    title='교정강도(Maintenance)별 LLM 모델 배정 현황',
    filename='maintenance_distribution.png'
)

In [ ]:
df['field'].value_counts(normalize=True)

In [ ]:
pd.crosstab(df['event_name'], df['total_paraphrasing_sentence_count'])

In [ ]:
pd.crosstab(df['field'], df['maintenance'])